In [1]:
import argparse
import os,cv2
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
fsc_test={}

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

for i in os.listdir('/kaggle/input/fake-scene-dataset/test'):
    fsc_test[i]=transform(Image.open(f'/kaggle/input/fake-scene-dataset/test/{i}'))

print(fsc_test["21.jpg"].shape)
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

torch.Size([3, 256, 256])


In [5]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1280, 2),
        )
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return torch.argmax(x, dim=1)

In [6]:
EFF_NET = EffnetModel().float().to(device)
#EFF_NET

#EFF_NET.load_state_dict(torch.load("/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/0.00015554654237348586 87.0.pth",weights_only=False,map_location=torch.device('cpu')))


Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:01<00:00, 157MB/s]  


In [7]:
fsc_submission=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col ="image")#.to_dict('list')
sub={}

for j in ['1.2919126675114967e-05 83.0','1.6423231500084512e-05 82.0','2.804919859045185e-05 83.0',
          '3.1164650863502175e-05 80.0','4.013385114376433e-05 88.0','5.1480830734362826e-05 82.0',
          '5.5708540457999334e-05 84.0','6.79657023283653e-05 84.0',
          '7.38521121093072e-05 85.0','7.9126468335744e-05 85.0','8.835738844936714e-05 84.0',
          '0.00010772716632345691 81.0','0.00011623594764387235 80.0','0.0001519525976618752 86.0']:
    z_add,z_total=0,0
    EFF_NET.load_state_dict(torch.load(f"/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/{j}.pth",weights_only=False,map_location=torch.device('cpu')))
    for i in fsc_submission.index:
        img=fsc_test[i].reshape((1, 3, 256, 256)).float().to(device)
        #print(FSC_net(img))

        output = EFF_NET(img).cpu().detach().numpy()[0]
        if output==0:
            z_add+=1
        z_total+=1
        #fsc_submission.loc[i]['label']=output
        if i not in sub.keys():
            if output==0:
                sub[i]={0:1,
                       1:0}
            else:
                sub[i]={0:0,
                       1:1}
        else:
            if output==0:
                sub[i][0]+=1
            else:
                sub[i][1]+=1
    print(z_total,z_add)
    
for i in fsc_submission.index:
    sub[i]=list(sub[i].values())

180 28
180 33
180 26
180 32
180 28
180 37
180 39
180 33
180 37
180 33
180 44
180 30
180 29


In [20]:
z_add,z_total=0,0

for i in fsc_submission.index:
    if sub[i][0]>=5:
        fsc_submission.loc[i]['label']=0
        z_add+=1
    else:
        fsc_submission.loc[i]['label']=1
    z_total+=1
print(z_total,z_add)

pandas.DataFrame(sub)

180 33


/tmp/ipykernel_30/1849759973.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fsc_submission.loc[i]['label']=1
/tmp/ipykernel_30/1849759973.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fsc_submission.loc[i]['label']=0


,102.jpg,108.jpg,109.jpg,111.jpg,121.jpg,122.jpg,126.jpg,127.jpg,130.jpg,133.jpg,...,884.jpg,889.jpg,890.jpg,896.jpg,897.jpg,899.jpg,91.jpg,94.jpg,95.jpg,98.jpg
0,0,0,0,14,0,0,0,14,0,13,...,13,0,2,0,0,11,0,0,1,14
1,14,14,14,0,14,14,14,0,14,1,...,1,14,12,14,14,3,14,14,13,0


In [19]:
fsc_submission['image']=fsc_submission.index
pandas.DataFrame(fsc_submission).to_csv(f"submission_45.csv", index=False)